# <!-- TITLE --> [CETO1] - Baseline ML : reconstruire l'historique à partir du catalogue
<!-- DESC --> Les photos de l'historique, annotées par Miraceti, attribuées par le ML à un dauphin du catalogue

## Pourquoi cette baseline
Ce notebook montre ce qu'on obtient déjà **sans pipeline**, avec seulement les empreintes de référence et la bibliothèque `dorsalmatch`. C'est le niveau à battre : toute nouvelle donnée ou nouveau modèle se compare à ces chiffres.

## La question
L'historique, ce sont des photos **déjà identifiées** par Miraceti : chacune porte le `TUTR` de son dauphin. Le catalogue, c'est une photo par côté pour chaque dauphin connu.

**Si on cache les annotations de l'historique, est-ce que le ML retrouve le bon dauphin dans le catalogue ?** Comme les annotations existent, on peut noter cette reconstruction.

Pour chaque photo (puis pour chaque dauphin), le bon dauphin tombe dans un de ces trois cas :

| Cas | Ce que ça veut dire pour Miraceti |
|---|---|
| **bien rangé** : rang 1 | le premier candidat proposé est le bon |
| **dans le top 10, mal rangé** : rang 2 à 10 | le bon est dans la liste, mais il faut descendre |
| **hors du top 10** | le bon n'est pas dans ce que Miraceti regarde |

## Les données
Les empreintes et les photos prêtes se récupèrent avec `notebooks/data_livraison.ipynb`, dans la partie ML et MLOps (décommenter le téléchargement des photos pour le Step 5).

## Step 1 - Initialisation

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import dorsalmatch  # noqa: E402

# CONSTANTES
DATA = ROOT / "data"
HISTORIQUE = "2026-10-29_miraceti"
CATALOGUE = "2026-11-06_catalogue_tt"
TOP = 10
MODEL = "cetaceanet"
SIDES = ("D", "G")
NEAR_IDENTICAL = 0.97  # au-delà, c'est la même photo
CASES = ["bien rangé", f"top {TOP}, mal rangé", f"hors top {TOP}"]

## Step 2 - Les données

In [ ]:
def load(kind, delivery, data=DATA):
    folder = data / "derived" / "embeddings" / kind / delivery
    # keep_default_na=False : un côté vide reste "", pas NaN
    photos = pd.read_csv(folder / "photos.csv", dtype=str, keep_default_na=False)
    return photos.to_dict("records"), np.load(folder / f"{MODEL}.npy")


# Toutes les photos contre un côté du catalogue : le meilleur cosinus de chaque dauphin
def best_cosines(photos, embeddings, catalogue, catalogue_embeddings, side):
    lines = [line for line, photo in enumerate(photos) if photo["cote"] in SIDES]
    catalogue_lines = [line for line, photo in enumerate(catalogue) if photo["cote"] == side]
    if not lines or not catalogue_lines:
        return {}, set()

    best, identical = {}, set()
    results = dorsalmatch.search(embeddings[lines], catalogue_embeddings[catalogue_lines], k=len(catalogue_lines))
    for closest in results:
        for position, cosine in closest:
            individual = catalogue[catalogue_lines[position]]["individu"]
            if cosine >= NEAR_IDENTICAL:
                identical.add(individual)
                continue
            best[individual] = max(best.get(individual, -1.0), cosine)
    return best, identical


# Score d'un dauphin : le meilleur cosinus, tous côtés confondus
def rank(photos, embeddings, catalogue, catalogue_embeddings):
    by_side, identical = {}, set()
    for side in SIDES:
        by_side[side], found = best_cosines(photos, embeddings, catalogue, catalogue_embeddings, side)
        identical |= found

    candidates = []
    for individual in set(by_side["D"]) | set(by_side["G"]):
        scores = {side: by_side[side][individual] for side in SIDES if individual in by_side[side]}
        candidates.append({"individu": individual, "score": float(max(scores.values())), **scores})
    return sorted(candidates, key=lambda candidate: (-candidate["score"], candidate["individu"])), identical


In [ ]:
history, history_embeddings = load("historique", HISTORIQUE, DATA)
catalogue, catalogue_embeddings = load("catalogue", CATALOGUE, DATA)

print(f"Historique : {len(history)} photos, {len({photo['individu'] for photo in history})} dauphins")
# Sans la photo au nom illisible : elle n'a pas de côté, ce n'est pas un dauphin
print(f"Catalogue  : {len(catalogue)} photos, {len({photo['individu'] for photo in catalogue if photo['cote']})} dauphins")

### 2.1 - Les photos qu'on ne peut pas noter

Le catalogue a été construit à partir de l'historique : beaucoup de photos de l'historique **sont** la photo du catalogue. Le ML retrouverait sa propre photo, ce qui ne prouve rien.

Les paires de photos presque identiques (cosinus ≥ `NEAR_IDENTICAL`) ne comptent donc pas dans le score. Une photo de l'historique dont la seule photo au catalogue, du même côté, est elle-même n'a rien à retrouver : elle est mise de côté.

In [ ]:
def findable(line):
    photo = history[line]
    same = [index for index, other in enumerate(catalogue)
            if other["individu"] == photo["individu"] and other["cote"] == photo["cote"]]
    if not same:
        return "côté absent du catalogue"
    if (catalogue_embeddings[same] @ history_embeddings[line] >= NEAR_IDENTICAL).all():
        return "c'est la photo du catalogue"
    return None


reasons = [findable(line) for line in range(len(history))]
for reason in sorted({reason for reason in reasons if reason}):
    print(f"{reasons.count(reason):>4} photos mises de côté : {reason}")
print(f"{reasons.count(None):>4} photos notées")

## Step 3 - La reconstruction

Pour chaque photo notée, le ML classe les dauphins du catalogue du plus au moins ressemblant, avec `dorsalmatch.search` et la fonction `rank` du Step 2, **comme dans la démo** : tous côtés confondus, meilleur cosinus par dauphin.

Puis, **par dauphin** : toutes ses photos de l'historique ensemble, les deux côtés combinés (meilleur cosinus), comme ce que Miraceti voit pour un individu d'un best-of.

In [ ]:
def reconstruct(individual, lines):
    candidates, _ = rank([history[line] for line in lines], history_embeddings[lines], catalogue, catalogue_embeddings)
    names = [candidate["individu"] for candidate in candidates]
    position = names.index(individual) if individual in names else None
    return {
        "individu": individual,
        "lines": lines,
        "candidats": candidates,
        "rang": position + 1 if position is not None else None,
        "score_bon": candidates[position]["score"] if position is not None else None,
        "score_premier_faux": next(candidate["score"] for candidate in candidates if candidate["individu"] != individual),
    }


def case(result):
    if result["rang"] == 1:
        return CASES[0]
    if result["rang"] and result["rang"] <= TOP:
        return CASES[1]
    return CASES[2]


by_photo = [reconstruct(history[line]["individu"], [line]) for line in range(len(history)) if reasons[line] is None]

lines_by_dolphin = {}
for line in range(len(history)):
    if reasons[line] is None:
        lines_by_dolphin.setdefault(history[line]["individu"], []).append(line)
by_dolphin = [reconstruct(individual, lines) for individual, lines in sorted(lines_by_dolphin.items())]

print(f"{len(by_photo)} photos et {len(by_dolphin)} dauphins reconstruits")

## Step 4 - Quanti

### 4.1 - Combien de bien rangés, de mal rangés, de ratés ?

In [ ]:
def counts(results, title):
    print(title)
    for name in CASES:
        number = sum(case(result) == name for result in results)
        print(f"  {name:<22} {number:>4}  ({number / len(results):.0%})")


counts(by_photo, f"Par photo ({len(by_photo)})")
counts(by_dolphin, f"Par dauphin ({len(by_dolphin)})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (results, title) in zip(axes, [(by_photo, "par photo"), (by_dolphin, "par dauphin")]):
    numbers = [sum(case(result) == name for result in results) for name in CASES]
    bars = ax.bar(CASES, numbers, color=["#2e7d32", "#f9a825", "#c62828"])
    ax.bar_label(bars, [f"{number} ({number / len(results):.0%})" for number in numbers])
    ax.set_title(title)
    ax.set_ylim(0, max(numbers) * 1.2)
plt.tight_layout()
plt.show()

### 4.2 - Quand il est hors du top 10, à quel rang est-il ?

Juste après la liste, ou complètement perdu ?

In [ ]:
missed = [result for result in by_photo if case(result) == CASES[2]]
print(f"{len(missed)} photos hors du top {TOP} :")
print(f"  rang médian du bon dauphin : {np.median([result['rang'] for result in missed]):.0f}")
for limit in (20, 50, 100):
    print(f"  dans le top {limit:<4} {sum(result['rang'] <= limit for result in missed)}")

### 4.3 - Les scores

Le score est le meilleur cosinus entre les empreintes des dorsales, tous côtés confondus : plus il est proche de 1, plus les dorsales se ressemblent.

Pour chaque photo : le score du **bon dauphin**, et celui du **premier faux candidat**. Si les deux se chevauchent, aucun seuil ne peut dire « c'est lui » : c'est à Miraceti de regarder.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
bins = np.linspace(0.2, 1, 33)
for ax, name in zip(axes, CASES):
    results = [result for result in by_photo if case(result) == name]
    ax.hist([result["score_bon"] for result in results], bins=bins, alpha=0.6, label="bon dauphin")
    ax.hist([result["score_premier_faux"] for result in results], bins=bins, alpha=0.6, label="premier faux")
    ax.set_title(f"{name} ({len(results)} photos)")
    ax.set_xlabel("score")
axes[0].set_ylabel("photos")
axes[0].legend()
plt.tight_layout()
plt.show()

for name in CASES:
    results = [result for result in by_photo if case(result) == name]
    print(f"{name:<22} score du bon : médiane {np.median([r['score_bon'] for r in results]):.2f}   "
          f"premier faux : médiane {np.median([r['score_premier_faux'] for r in results]):.2f}")

### 4.4 - Droite ou gauche ?

In [ ]:
for side, name in (("D", "droite"), ("G", "gauche")):
    results = [result for result in by_photo if history[result["lines"][0]]["cote"] == side]
    shares = "   ".join(f"{case_name} {sum(case(r) == case_name for r in results) / len(results):.0%}" for case_name in CASES)
    print(f"{name:<7} {len(results):>4} photos   {shares}")

## Step 5 - Quali

Une photo de l'historique, face aux 10 dauphins proposés par le ML : **vert**, le bon dauphin ; **rouge**, un autre. Le chiffre est le score.

Pour chaque cas, regarder ce qui se passe : photo floue, dorsale de loin, angle, dorsale peu marquée…

**Les photos de Miraceti ne sont pas publiques : effacer les sorties de cette partie avant de commiter ce notebook.** Les chiffres et les graphiques des parties précédentes peuvent rester.

In [ ]:
def photo_path(kind, delivery, photo):
    return DATA / "curated" / kind / "photos" / delivery / photo["individu"] / photo["fichier_origine"]


def catalogue_photo(individual, side):
    return next((photo for photo in catalogue if photo["individu"] == individual and photo["cote"] == side), None)


def small(path):
    image = Image.open(path).convert("RGB")
    image.thumbnail((400, 400))
    return image


def show(result):
    query = history[result["lines"][0]]
    fig = plt.figure(figsize=(14, 4.2), dpi=80)
    grid = fig.add_gridspec(2, 6, width_ratios=[2.2, 1, 1, 1, 1, 1], wspace=0.1, hspace=0.5, top=0.8, bottom=0.02)
    ax = fig.add_subplot(grid[:, 0])
    ax.imshow(small(photo_path("historique", HISTORIQUE, query)))
    ax.set_title(f"{query['individu']} · {query['cote']} · rang {result['rang']}", loc="left", fontsize=11)
    ax.axis("off")
    for position, candidate in enumerate(result["candidats"][:TOP]):
        good = candidate["individu"] == query["individu"]
        ax = fig.add_subplot(grid[position // 5, 1 + position % 5])
        photo = catalogue_photo(candidate["individu"], query["cote"])
        if photo:
            ax.imshow(small(photo_path("catalogue", CATALOGUE, photo)))
        ax.set_title(f"{candidate['individu']} · {candidate['score']:.2f}", fontsize=9, color="green" if good else "red")
        ax.axis("off")
    plt.show()


RANDOM = np.random.default_rng(0)


def examples(name, number=3):
    results = [result for result in by_photo if case(result) == name]
    for index in RANDOM.choice(len(results), size=min(number, len(results)), replace=False):
        show(results[index])

### 5.1 - Bien rangé

In [ ]:
examples(CASES[0])

### 5.2 - Dans le top 10, mal rangé

In [ ]:
examples(CASES[1])

### 5.3 - Hors du top 10

In [ ]:
examples(CASES[2])

## Step 6 - Ce qu'on retient

In [ ]:
for results, title in ((by_photo, "photos"), (by_dolphin, "dauphins")):
    shares = [sum(case(result) == name for result in results) / len(results) for name in CASES]
    print(f"Sur {len(results)} {title} de l'historique, le ML range le bon dauphin du catalogue :")
    print(f"  premier pour {shares[0]:.0%}, dans le top {TOP} mais plus bas pour {shares[1]:.0%}, hors du top {TOP} pour {shares[2]:.0%}")